# Simplified MOT Disc Sampling Geometry

This notebook visualizes one incident sampling disc for the simplified MOT model, draws the cooling beams with their full `12.7 mm` diameter, and lets you test trajectories launched from that disc geometry.

In [ ]:
%matplotlib widget

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pmot.forces import AtomState
from pmot.mot import default_anti_helmholtz_config
from pmot.mot_simple import (
    build_simple_mot_beams,
    default_simple_mot_apparatus,
    default_simple_mot_config,
    draw_simple_mot_beam_volumes,
    simulate_simple_mot_trajectory,
)
from pmot.mot_simple.sampling import (
    CaptureSearchConfig,
    build_incident_disc_from_angles,
    classify_trajectory,
)

APPARATUS = default_simple_mot_apparatus()
SIMPLE_CONFIG = default_simple_mot_config()
COIL_CONFIG = default_anti_helmholtz_config()
MOT_BEAMS = build_simple_mot_beams(APPARATUS, SIMPLE_CONFIG)
VELOCITY_SWEEP_M_PER_S = [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0, 20.0]


In [ ]:
def build_launch_point(theta_deg, phi_deg, radial_distance_mm, theta_prime_deg, impact_parameter_mm):
    theta_rad = np.deg2rad(theta_deg)
    phi_rad = np.deg2rad(phi_deg)
    theta_prime_rad = np.deg2rad(theta_prime_deg)
    disc = build_incident_disc_from_angles(
        disc_index=0,
        radial_distance_m=1e-3 * radial_distance_mm,
        theta_rad=theta_rad,
        phi_rad=phi_rad,
    )
    offset = (
        impact_parameter_mm * 1e-3 * np.cos(theta_prime_rad) * np.asarray(disc.basis_u, dtype=float)
        + impact_parameter_mm * 1e-3 * np.sin(theta_prime_rad) * np.asarray(disc.basis_v, dtype=float)
    )
    initial_position_m = np.asarray(disc.center_position_m, dtype=float) + offset
    launch_direction = -np.asarray(disc.center_position_m, dtype=float)
    launch_direction = launch_direction / np.linalg.norm(launch_direction)
    return disc, tuple(initial_position_m.tolist()), tuple(launch_direction.tolist())


def trajectory_status_style(classification):
    if classification.trapped:
        return 'TRAPPED', '#15803d'
    if classification.termination_reason == 'escaped':
        return 'ESCAPED', '#b91c1c'
    return classification.termination_reason.upper(), '#c2410c'


def make_search_config(duration_ms, time_step_us):
    return CaptureSearchConfig(
        max_simulation_time_s=1e-3 * duration_ms,
        time_step_s=1e-6 * time_step_us,
        disc_radius_m=12.0e-3,
    )


## Disc Geometry

In [ ]:
geometry_theta = widgets.FloatSlider(description='theta [deg]', min=0.0, max=90.0, step=1.0, value=35.0)
geometry_phi = widgets.FloatSlider(description='phi [deg]', min=0.0, max=90.0, step=1.0, value=20.0)
geometry_d = widgets.FloatSlider(description='d [mm]', min=5.0, max=30.0, step=0.5, value=15.0)
geometry_smax = widgets.FloatSlider(description='s_max [mm]', min=1.0, max=15.0, step=0.5, value=12.0)
geometry_rings = widgets.IntSlider(description='rings', min=3, max=20, step=1, value=8)
geometry_seed = widgets.IntSlider(description='seed', min=0, max=100, step=1, value=0)
geometry_button = widgets.Button(description='Update Geometry', button_style='primary')
geometry_output = widgets.Output()

def draw_disc_geometry(_=None):
    with geometry_output:
        geometry_output.clear_output(wait=True)
        rng = np.random.default_rng(geometry_seed.value)
        disc, _, launch_direction = build_launch_point(
            geometry_theta.value,
            geometry_phi.value,
            geometry_d.value,
            0.0,
            0.0,
        )
        figure = plt.figure(figsize=(9.0, 7.5), constrained_layout=True)
        figure.patch.set_facecolor('#fbfaf6')
        axis = figure.add_subplot(111, projection='3d')
        axis.set_facecolor('#fbfaf6')
        draw_simple_mot_beam_volumes(axis, MOT_BEAMS)

        basis_u_mm = 1e3 * np.asarray(disc.basis_u, dtype=float)
        basis_v_mm = 1e3 * np.asarray(disc.basis_v, dtype=float)
        center_mm = 1e3 * np.asarray(disc.center_position_m, dtype=float)
        radius_mm = geometry_smax.value

        point_positions_mm = [center_mm]
        if geometry_rings.value > 1:
            radii_mm = np.linspace(0.0, radius_mm, geometry_rings.value)
            angular_count = max(8, geometry_rings.value)
            for ring_radius_mm in radii_mm[1:]:
                phase = float(rng.uniform(0.0, 2.0 * np.pi))
                angles = np.linspace(0.0, 2.0 * np.pi, angular_count, endpoint=False) + phase
                for angle in angles:
                    point_positions_mm.append(
                        center_mm
                        + ring_radius_mm * np.cos(angle) * basis_u_mm
                        + ring_radius_mm * np.sin(angle) * basis_v_mm
                    )

        point_positions_mm = np.asarray(point_positions_mm, dtype=float)
        rim_angles = np.linspace(0.0, 2.0 * np.pi, 240)
        rim_mm = center_mm[None, :] + radius_mm * (
            np.cos(rim_angles)[:, None] * basis_u_mm[None, :] + np.sin(rim_angles)[:, None] * basis_v_mm[None, :]
        )

        axis.plot(rim_mm[:, 0], rim_mm[:, 1], rim_mm[:, 2], color='#7c3aed', linewidth=2.0, alpha=0.9)
        axis.scatter([0.0], [0.0], [0.0], color='#111827', s=40, label='trap center')
        axis.scatter([center_mm[0]], [center_mm[1]], [center_mm[2]], color='#0f766e', s=55, label='disc center')
        axis.scatter(point_positions_mm[:, 0], point_positions_mm[:, 1], point_positions_mm[:, 2], color='#059669', s=14, alpha=0.85, label='disc sample points')
        axis.quiver(center_mm[0], center_mm[1], center_mm[2], 14.0 * launch_direction[0], 14.0 * launch_direction[1], 14.0 * launch_direction[2], color='#c2410c', linewidth=2.0, arrow_length_ratio=0.14)
        axis.set_title(
            'Disc Geometry with Cooling Beam Volumes\n'
            f'theta={geometry_theta.value:.1f} deg, phi={geometry_phi.value:.1f} deg, d={geometry_d.value:.1f} mm, s_max={geometry_smax.value:.1f} mm'
        )
        axis.set_xlabel('x [mm]')
        axis.set_ylabel('y [mm]')
        axis.set_zlabel('z [mm]')
        axis.set_xlim(-25.0, 25.0)
        axis.set_ylim(-25.0, 25.0)
        axis.set_zlim(-25.0, 25.0)
        axis.set_box_aspect((1.0, 1.0, 1.0))
        axis.legend(loc='best')
        plt.show()

geometry_button.on_click(draw_disc_geometry)
display(widgets.VBox([
    widgets.HBox([geometry_theta, geometry_phi]),
    widgets.HBox([geometry_d, geometry_smax]),
    widgets.HBox([geometry_rings, geometry_seed]),
    geometry_button,
    geometry_output,
]))
draw_disc_geometry()


## Single Trajectory

In [ ]:
single_theta = widgets.FloatSlider(description='theta [deg]', min=0.0, max=90.0, step=1.0, value=35.0)
single_phi = widgets.FloatSlider(description='phi [deg]', min=0.0, max=90.0, step=1.0, value=20.0)
single_d = widgets.FloatSlider(description='d [mm]', min=5.0, max=30.0, step=0.5, value=15.0)
single_theta_prime = widgets.FloatSlider(description="theta' [deg]", min=0.0, max=360.0, step=1.0, value=0.0)
single_s = widgets.FloatSlider(description='s [mm]', min=0.0, max=12.0, step=0.25, value=3.0)
single_v0 = widgets.FloatSlider(description='v0 [m/s]', min=1.0, max=30.0, step=0.5, value=5.0)
single_duration = widgets.FloatSlider(description='T [ms]', min=1.0, max=80.0, step=1.0, value=25.0)
single_dt = widgets.FloatSlider(description='dt [us]', min=1.0, max=20.0, step=1.0, value=5.0)
single_button = widgets.Button(description='Run Single Trajectory', button_style='primary')
single_output = widgets.Output()

def run_single_trajectory(_=None):
    with single_output:
        single_output.clear_output(wait=True)
        _, initial_position_m, launch_direction = build_launch_point(
            single_theta.value,
            single_phi.value,
            single_d.value,
            single_theta_prime.value,
            single_s.value,
        )
        initial_velocity = tuple((single_v0.value * np.asarray(launch_direction, dtype=float)).tolist())
        search_config = make_search_config(single_duration.value, single_dt.value)
        trajectory = simulate_simple_mot_trajectory(
            beams=MOT_BEAMS,
            initial_state=AtomState(position_m=initial_position_m, velocity_m_per_s=initial_velocity),
            duration_s=1e-3 * single_duration.value,
            time_step_s=1e-6 * single_dt.value,
            coil_config=COIL_CONFIG,
            simple_config=SIMPLE_CONFIG,
        )
        classification = classify_trajectory(
            MOT_BEAMS,
            type('LaunchPoint', (), {
                'initial_position_m': initial_position_m,
                'incident_unit_vector': launch_direction,
            })(),
            single_v0.value,
            COIL_CONFIG,
            SIMPLE_CONFIG,
            search_config,
        )
        status_text, status_color = trajectory_status_style(classification)
        positions_mm = 1e3 * np.asarray(trajectory.positions_m, dtype=float)
        figure = plt.figure(figsize=(9.0, 7.5), constrained_layout=True)
        figure.patch.set_facecolor('#fbfaf6')
        axis = figure.add_subplot(111, projection='3d')
        axis.set_facecolor('#fbfaf6')
        draw_simple_mot_beam_volumes(axis, MOT_BEAMS)
        axis.plot(positions_mm[:, 0], positions_mm[:, 1], positions_mm[:, 2], color='#0f766e', linewidth=2.2)
        axis.scatter([positions_mm[0, 0]], [positions_mm[0, 1]], [positions_mm[0, 2]], color='#b91c1c', s=50, label='start')
        axis.scatter([positions_mm[-1, 0]], [positions_mm[-1, 1]], [positions_mm[-1, 2]], color='#111827', s=50, label='end')
        axis.scatter([0.0], [0.0], [0.0], color='#7c3aed', s=36, label='trap center')
        axis.text2D(0.03, 0.95, status_text, transform=axis.transAxes, color=status_color, fontsize=13, fontweight='bold')
        axis.set_title(
            'Single Launch Trajectory\n'
            f'd={single_d.value:.1f} mm, theta={single_theta.value:.1f} deg, phi={single_phi.value:.1f} deg, theta\'={single_theta_prime.value:.1f} deg, s={single_s.value:.1f} mm, v0={single_v0.value:.1f} m/s'
        )
        axis.set_xlabel('x [mm]')
        axis.set_ylabel('y [mm]')
        axis.set_zlabel('z [mm]')
        axis.set_xlim(-25.0, 25.0)
        axis.set_ylim(-25.0, 25.0)
        axis.set_zlim(-25.0, 25.0)
        axis.set_box_aspect((1.0, 1.0, 1.0))
        axis.legend(loc='best')
        plt.show()

single_button.on_click(run_single_trajectory)
display(widgets.VBox([
    widgets.HBox([single_theta, single_phi, single_d]),
    widgets.HBox([single_theta_prime, single_s, single_v0]),
    widgets.HBox([single_duration, single_dt]),
    single_button,
    single_output,
]))
run_single_trajectory()


## 3x3 Velocity Sweep

In [ ]:
grid_theta = widgets.FloatSlider(description='theta [deg]', min=0.0, max=90.0, step=1.0, value=35.0)
grid_phi = widgets.FloatSlider(description='phi [deg]', min=0.0, max=90.0, step=1.0, value=20.0)
grid_d = widgets.FloatSlider(description='d [mm]', min=5.0, max=30.0, step=0.5, value=15.0)
grid_theta_prime = widgets.FloatSlider(description="theta' [deg]", min=0.0, max=360.0, step=1.0, value=0.0)
grid_s = widgets.FloatSlider(description='s [mm]', min=0.0, max=12.0, step=0.25, value=3.0)
grid_duration = widgets.FloatSlider(description='T [ms]', min=1.0, max=80.0, step=1.0, value=25.0)
grid_dt = widgets.FloatSlider(description='dt [us]', min=1.0, max=20.0, step=1.0, value=5.0)
grid_button = widgets.Button(description='Run Velocity Sweep', button_style='primary')
grid_output = widgets.Output()

def run_velocity_grid(_=None):
    with grid_output:
        grid_output.clear_output(wait=True)
        _, initial_position_m, launch_direction = build_launch_point(
            grid_theta.value,
            grid_phi.value,
            grid_d.value,
            grid_theta_prime.value,
            grid_s.value,
        )
        search_config = make_search_config(grid_duration.value, grid_dt.value)
        figure = plt.figure(figsize=(12.5, 12.5), constrained_layout=True)
        figure.patch.set_facecolor('#fbfaf6')
        axes = [figure.add_subplot(3, 3, index + 1, projection='3d') for index in range(9)]
        for axis, speed_m_per_s in zip(axes, VELOCITY_SWEEP_M_PER_S):
            initial_velocity = tuple((speed_m_per_s * np.asarray(launch_direction, dtype=float)).tolist())
            trajectory = simulate_simple_mot_trajectory(
                beams=MOT_BEAMS,
                initial_state=AtomState(position_m=initial_position_m, velocity_m_per_s=initial_velocity),
                duration_s=1e-3 * grid_duration.value,
                time_step_s=1e-6 * grid_dt.value,
                coil_config=COIL_CONFIG,
                simple_config=SIMPLE_CONFIG,
            )
            classification = classify_trajectory(
                MOT_BEAMS,
                type('LaunchPoint', (), {
                    'initial_position_m': initial_position_m,
                    'incident_unit_vector': launch_direction,
                })(),
                speed_m_per_s,
                COIL_CONFIG,
                SIMPLE_CONFIG,
                search_config,
            )
            status_text, status_color = trajectory_status_style(classification)
            positions_mm = 1e3 * np.asarray(trajectory.positions_m, dtype=float)
            axis.set_facecolor('#fbfaf6')
            draw_simple_mot_beam_volumes(axis, MOT_BEAMS)
            axis.plot(positions_mm[:, 0], positions_mm[:, 1], positions_mm[:, 2], color='#0f766e', linewidth=1.8)
            axis.scatter([positions_mm[0, 0]], [positions_mm[0, 1]], [positions_mm[0, 2]], color='#b91c1c', s=28)
            axis.scatter([positions_mm[-1, 0]], [positions_mm[-1, 1]], [positions_mm[-1, 2]], color='#111827', s=28)
            axis.scatter([0.0], [0.0], [0.0], color='#7c3aed', s=18)
            axis.text2D(0.05, 0.92, status_text, transform=axis.transAxes, color=status_color, fontsize=10, fontweight='bold')
            axis.set_title(f'v0 = {speed_m_per_s:.0f} m/s', fontsize=10)
            axis.set_xlabel('x [mm]')
            axis.set_ylabel('y [mm]')
            axis.set_zlabel('z [mm]')
            axis.set_xlim(-25.0, 25.0)
            axis.set_ylim(-25.0, 25.0)
            axis.set_zlim(-25.0, 25.0)
            axis.set_box_aspect((1.0, 1.0, 1.0))
        figure.suptitle(
            'Velocity Sweep for Fixed Disc Geometry\n'
            f'd={grid_d.value:.1f} mm, theta={grid_theta.value:.1f} deg, phi={grid_phi.value:.1f} deg, theta\'={grid_theta_prime.value:.1f} deg, s={grid_s.value:.1f} mm',
            fontsize=14,
        )
        plt.show()

grid_button.on_click(run_velocity_grid)
display(widgets.VBox([
    widgets.HBox([grid_theta, grid_phi, grid_d]),
    widgets.HBox([grid_theta_prime, grid_s]),
    widgets.HBox([grid_duration, grid_dt]),
    grid_button,
    grid_output,
]))
run_velocity_grid()
